In [1]:
import pandas as pd
from pathlib import Path


In [2]:
transients = pd.read_parquet(
    "snia_results.parquet",
    engine="fastparquet",
)

In [3]:
transients

,diaObjectId,tns_name,z,catalogue_object_id,tns_prefix,type,ra,decl,absMag,nDiaSources
0,170019717277810735,2026ctw,0.150000,"10000,342494,11570",SN,SN Ia,151.740293,1.927064,NaN,754
1,170028486191218743,2026qrh,0.100000,"10000,81273,866",SN,SN Ia,64.185342,-48.943229,NaN,88
2,170028500610711681,2024xom,0.044167,CGCG036-024,SN,SN Ia,150.746125,2.336469,NaN,165
3,170028510695916335,2025mbp,0.046160,2MASXJ10062208+0109317,SN,SN Ia,151.592617,1.158385,NaN,273
4,170028520834072659,2026fgl,0.164000,WISEAJ095652.35+032811.7,SN,SN Ia,149.218165,3.469919,-16.5385,19
...,...,...,...,...,...,...,...,...,...,...
122,170661828398940941,2025bbx,0.067000,14470795-3930390,SN,SN Ia,221.785403,-39.509297,NaN,3
123,313893021499785278,2026ezw,0.131000,SDSSJ095832.59+011610.2,SN,SN Ia,149.635687,1.269344,NaN,20
124,313932566867476549,2026qrh,0.100000,"10000,81273,866",SN,SN Ia,64.185342,-48.943229,NaN,9
125,313972182793322538,2026fgl,0.164000,WISEAJ095652.35+032811.7,SN,SN Ia,149.218165,3.469919,-16.2020,17


In [4]:
transients = (
    transients[["diaObjectId", "tns_name", "ra", "decl"]]
    .rename(columns={"decl": "dec"})
    .dropna()
    .reset_index(drop=True)
    .copy()
)

transients["tns_name"] = transients["tns_name"].astype(str)

print(f"Associating {len(transients)} transients")
display(transients)

Associating 127 transients


,diaObjectId,tns_name,ra,dec
0,170019717277810735,2026ctw,151.740293,1.927064
1,170028486191218743,2026qrh,64.185342,-48.943229
2,170028500610711681,2024xom,150.746125,2.336469
3,170028510695916335,2025mbp,151.592617,1.158385
4,170028520834072659,2026fgl,149.218165,3.469919
...,...,...,...,...
122,170661828398940941,2025bbx,221.785403,-39.509297
123,313893021499785278,2026ezw,149.635687,1.269344
124,313932566867476549,2026qrh,64.185342,-48.943229
125,313972182793322538,2026fgl,149.218165,3.469919


In [5]:
import inspect
import textwrap
import astro_prost.helpers as helpers

func = helpers.build_panstarrs_candidates
source = textwrap.dedent(inspect.getsource(func))
lines = source.splitlines()

for i, line in enumerate(lines):
    if line.lstrip().startswith("temp_mag_r_std[temp_mag_r_std >"):
        indent = line[:len(line) - len(line.lstrip())]
        lines.insert(
            i,
            indent + "temp_mag_r_std = temp_mag_r_std.copy()",
        )
        break
else:
    raise RuntimeError("Could not locate the failing line; no patch applied.")

# Compile the patched function using the same module globals.
namespace = {}
exec(compile("\n".join(lines), "<prost_panstarrs_fix>", "exec"),
     helpers.__dict__, namespace)

# Update the existing function so existing references also use the fix.
func.__code__ = namespace[func.__name__].__code__

print("Applied writable-array fix to PROST's Pan-STARRS function.")

Applied writable-array fix to PROST's Pan-STARRS function.


In [ ]:
import inspect
import textwrap
import traceback

import pandas as pd
import astro_prost.helpers as prost_helpers
from scipy.stats import gamma, uniform
from astro_prost import associate_sample

# Temporary read-only fix without editing helpers.py.
if not getattr(prost_helpers, "_shape_r_copy_patch_applied", False):
    source = textwrap.dedent(
        inspect.getsource(prost_helpers.calc_shape_props_decals)
    )

    old = 'candidate_hosts["shape_r"].values'
    new = 'candidate_hosts["shape_r"].to_numpy(copy=True)'

    if old not in source:
        raise RuntimeError("Expected shape_r code not found.")

    source = source.replace(old, new)
    exec(
        compile(source, "<prost_notebook_patch>", "exec"),
        prost_helpers.__dict__,
    )
    prost_helpers._shape_r_copy_patch_applied = True

# Collect results in memory.
results = []
failed = []

for i in range(len(transients)):
    one_transient = transients.iloc[[i]].copy().reset_index(drop=True)
    name = str(one_transient.iloc[0]["tns_name"])

    try:
        result = associate_sample(
        one_transient,
        catalogs=["decals", "panstarrs", "skymapper"],
        name_col="tns_name",
        coord_cols=("ra", "dec"),
        priors={"offset": uniform(loc=0, scale=10)},
        likes={"offset": gamma(a=0.75)},
        n_samples=1000,  # Match the 1,000 samples used internally.
        parallel=False,
        save=False,
        )
    

        if isinstance(result, Exception):
            raise result

        if not isinstance(result, pd.DataFrame):
            raise TypeError(
                f"Expected DataFrame; got {type(result).__name__}"
            )

        if result.empty:
            raise ValueError("PROST returned an empty DataFrame.")

        results.append(result)
        print(f"{i + 1}/{len(transients)} done: {name}")

    except Exception as exc:
        failed.append({"tns_name": name, "error": repr(exc)})
        print(f"{i + 1}/{len(transients)} failed: {name}: {exc}")
        traceback.print_exc()

hosts = pd.concat(results, ignore_index=True) if results else pd.DataFrame()
failed_df = pd.DataFrame(failed, columns=["tns_name", "error"])

display(hosts)
display(failed_df)

2026-10-14 11:31:53,054- INFO - Conditioning association on the following properties: ['offset']
2026-10-14 11:31:55,506- INFO - Loaded GLADE+ catalog.
2026-10-14 11:31:55,620- INFO - 

Associating 2026ctw at RA, DEC = 151.740293, 1.927064 (redshift nan)
2026-10-14 11:31:57,562- INFO - Association successful!
2026-10-14 11:31:57,582- INFO - 
    
Properties of best host (in decals dr9) for 2026ctw:
    objID: 9906622211952776
    R.A. (deg): 151.740166
    Dec. (deg): 1.927030
    Posterior: 1.0000e+00
    Redshift: 0.7213 ± 0.3376 (PHOT)
    Absolute Magnitude: -20.3 ± 1.1 (r)
    Transient Offset: 0.5 ± 0.1 arcsec
    Transient Offset Posterior: 1.0000e+00
2026-10-14 11:31:57,583- INFO - 
    
Properties of 2nd best host (in decals dr9) for 2026ctw:
    Redshift: 1.0042 ± 0.1648 (PHOT)
    Absolute Magnitude: -19.8 ± 1.2 (r)
    Transient Offset: 292.4 ± 0.1 arcsec
    Transient Offset Posterior: 0.0000e+00
2026-10-14 11:31:57,583- INFO - Chosen galaxy has catalog ID of 9906622211952

1/127 done: 2026ctw


In [ ]:
results